# Score-Based IV Smile Completion for `dataset.csv`

This Kaggle notebook implements the paper pipeline from **Ying Kit Hui, “Volatility surface completion using score-based generative models”** and adapts it to this dataset.

The paper's exact framework is:

1. Treat the implied volatility surface as an image.
2. Train a **Noise Conditional Score Network** with denoising score matching.
3. Complete missing values by **Annealed Langevin Dynamics inpainting**.
4. Optionally guide sampling with no-arbitrage losses.

Important adaptation for this dataset: the paper works with full 2D strike × maturity surfaces. Your file has one expiry and two option-type smiles, so this notebook uses a `2 × 14` image per timestamp:

- row 0 = CE smile across strike rank
- row 1 = PE smile across strike rank

The notebook validates the score/inpainting method against a deterministic cross-section baseline before writing the final submission. If the learned model is worse on validation, the notebook reports that clearly and uses a safe blended fallback instead of blindly trusting the neural model.

In [ ]:
# ================================================================
# Kaggle setup
# ================================================================
# Expected input: a CSV named dataset.csv somewhere under /kaggle/input
# You can also upload dataset.csv to the notebook working directory.

import os, re, math, json, random, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import PchipInterpolator
from scipy.ndimage import gaussian_filter1d

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

# ---------------- User knobs ----------------
DATA_PATH = None  # leave None to auto-find dataset.csv
OUT_PREFIX = 'score_ncsn_iv'

# Training settings. Start here; increase epochs if Kaggle GPU time allows.
EPOCHS = 80
BATCH_SIZE = 64
LR = 2e-4
WEIGHT_DECAY = 1e-5

# Noise schedule for NCSN / ALD.
NUM_NOISE_LEVELS = 16
SIGMA_MIN = 0.01
SIGMA_MAX_MODE = 'data'  # 'data' chooses from empirical surface distance; or set SIGMA_MAX_FLOAT below
SIGMA_MAX_FLOAT = 1.0

# ALD inpainting settings.
ALD_STEPS_PER_NOISE = 35
ALD_EPS = 2e-5
ALD_RESTARTS = 4
USE_NO_ARB_PROXY = True  # paper uses no-arbitrage losses; here single-expiry proxy is butterfly/convexity only
NO_ARB_EPS = 1e-3
ACCEPT_UPPER = 1.2

# Validation masking.
VAL_FRAC = 0.20
VAL_MASK_REPEATS = 2
VAL_HIDE_FRAC = 0.20
EDGE_EXTRA_VALIDATION = True

# Final output behavior.
# If neural inpainting is not validated better than baseline, final output uses a conservative blend.
MAX_NEURAL_BLEND_IF_WORSE = 0.15

In [ ]:
# ================================================================
# Data loading and metadata parsing
# ================================================================

def auto_find_dataset():
    candidates = []
    for root in ['.', '/kaggle/input', '/kaggle/working']:
        rootp = Path(root)
        if rootp.exists():
            candidates.extend(rootp.rglob('dataset.csv'))
    if not candidates:
        raise FileNotFoundError('Could not find dataset.csv. Upload it to Kaggle input or working directory.')
    candidates = sorted(candidates, key=lambda p: len(str(p)))
    return candidates[0]

if DATA_PATH is None:
    DATA_PATH = auto_find_dataset()
else:
    DATA_PATH = Path(DATA_PATH)

print('Using DATA_PATH:', DATA_PATH)
df_raw = pd.read_csv(DATA_PATH)
print(df_raw.shape)
df_raw.head()

In [ ]:
PATTERN = re.compile(
    r'^(?P<underlying>[A-Z]+)'
    r'(?P<expiry>\d{2}[A-Z]{3}\d{2})'
    r'(?P<strike>\d+)'
    r'(?P<option_type>CE|PE)$'
)

def parse_metadata(df: pd.DataFrame) -> pd.DataFrame:
    recs = []
    for col in df.columns:
        if col in {'datetime','underlying_price','datetime_parsed'}:
            continue
        m = PATTERN.match(col)
        if m:
            d = m.groupdict()
            d['column'] = col
            d['strike'] = int(d['strike'])
            d['expiry_date'] = pd.to_datetime(d['expiry'], format='%d%b%y', errors='coerce')
            recs.append(d)
    meta = pd.DataFrame(recs)
    if meta.empty:
        raise ValueError('No option columns parsed. Check CSV headers.')
    return meta.sort_values(['option_type','strike','column']).reset_index(drop=True)

df = df_raw.copy()
df['datetime_parsed'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M', errors='coerce')
if df['datetime_parsed'].isna().any():
    raise ValueError('Some datetimes could not be parsed.')
df = df.sort_values('datetime_parsed').reset_index(drop=True)

meta = parse_metadata(df)
option_cols = meta['column'].tolist()
strike_map = dict(zip(meta['column'], meta['strike']))
type_map = dict(zip(meta['column'], meta['option_type']))
cols_by_type = {
    'CE': [c for c in option_cols if type_map[c] == 'CE'],
    'PE': [c for c in option_cols if type_map[c] == 'PE'],
}
print(meta.groupby('option_type')['column'].count())
print('Missing cells:', int(df[option_cols].isna().sum().sum()))
meta.head()

In [ ]:
# ================================================================
# Image representation: 2 x max_rank
# row 0 = CE, row 1 = PE
# ================================================================

MAX_W = max(len(cols_by_type['CE']), len(cols_by_type['PE']))
H, W = 2, MAX_W
ROW_OF_TYPE = {'CE':0, 'PE':1}
TYPE_OF_ROW = {0:'CE', 1:'PE'}

# Fixed image column order per option type.
rank_to_col = {}
col_to_pos = {}
for ot in ['CE','PE']:
    cols = sorted(cols_by_type[ot], key=lambda c: strike_map[c])
    for j, c in enumerate(cols):
        rank_to_col[(ot,j)] = c
        col_to_pos[c] = (ROW_OF_TYPE[ot], j)

valid_pixel = np.zeros((H,W), dtype=np.float32)
for c, (i,j) in col_to_pos.items():
    valid_pixel[i,j] = 1.0

def row_to_image(row: pd.Series) -> Tuple[np.ndarray, np.ndarray]:
    img = np.zeros((H,W), dtype=np.float32)
    mask = np.zeros((H,W), dtype=np.float32)
    for c in option_cols:
        i,j = col_to_pos[c]
        v = row[c]
        if pd.notna(v) and np.isfinite(v):
            img[i,j] = float(v)
            mask[i,j] = 1.0
    return img, mask

images = []
masks = []
for _, row in df.iterrows():
    im, ma = row_to_image(row)
    images.append(im)
    masks.append(ma)
images = np.stack(images)
masks = np.stack(masks)

print('images:', images.shape, 'masks:', masks.shape)
plt.figure(figsize=(10,2.4))
plt.imshow(images[0], aspect='auto')
plt.title('Example IV image, missing shown as 0 before normalization')
plt.yticks([0,1], ['CE','PE'])
plt.colorbar();

In [ ]:
# ================================================================
# Deterministic cross-section baseline used for:
# 1. pseudo-complete training images,
# 2. validation comparison,
# 3. safe fallback if score model is not better.
#
# It combines PCHIP interpolation for interior cells with quadratic/local
# extrapolation at edges, separately by timestamp and option type.
# ================================================================

def safe_iv(x):
    try:
        x = float(x)
    except Exception:
        return np.nan
    if not np.isfinite(x):
        return np.nan
    return max(x, 1e-6)

GLOBAL_MEDIAN_IV = float(pd.Series(df[option_cols].values.ravel()).dropna().median())

def baseline_fill_one_row(row: pd.Series) -> Dict[str, float]:
    pred = {}
    spot = float(row['underlying_price']) if pd.notna(row['underlying_price']) else np.nan
    for ot in ['CE','PE']:
        cols = sorted(cols_by_type[ot], key=lambda c: strike_map[c])
        x_all = np.array([strike_map[c] / spot for c in cols], dtype=float) if np.isfinite(spot) and spot > 0 else np.arange(len(cols), dtype=float)
        y_all = np.array([row[c] if pd.notna(row[c]) else np.nan for c in cols], dtype=float)
        obs = np.isfinite(y_all)
        if obs.sum() == 0:
            for c in cols:
                pred[c] = GLOBAL_MEDIAN_IV
            continue
        if obs.sum() == 1:
            for c in cols:
                pred[c] = safe_iv(y_all[obs][0])
            continue

        x_obs = x_all[obs]
        y_obs = y_all[obs]
        order = np.argsort(x_obs)
        x_obs, y_obs = x_obs[order], y_obs[order]
        # De-duplicate x if needed.
        _, unique_idx = np.unique(x_obs, return_index=True)
        x_obs, y_obs = x_obs[unique_idx], y_obs[unique_idx]

        use_pchip = len(x_obs) >= 3
        pchip = None
        if use_pchip:
            try:
                pchip = PchipInterpolator(x_obs, y_obs, extrapolate=False)
            except Exception:
                pchip = None

        # edge quadratic fit using all observed points, nearest weighted implicitly by low degree.
        deg = min(2, len(x_obs)-1)
        coef = np.polyfit(x_obs, y_obs, deg)

        for c, x in zip(cols, x_all):
            if pd.notna(row[c]):
                pred[c] = safe_iv(row[c])
                continue
            if pchip is not None and x_obs.min() <= x <= x_obs.max():
                v = float(pchip(x))
            else:
                v = float(np.polyval(coef, x))
            pred[c] = safe_iv(v)
    return pred

def baseline_complete_df(source_df: pd.DataFrame) -> pd.DataFrame:
    out = source_df.copy()
    for idx, row in tqdm(source_df.iterrows(), total=len(source_df), desc='Baseline completing rows'):
        preds = baseline_fill_one_row(row)
        for c in option_cols:
            if pd.isna(out.at[idx,c]):
                out.at[idx,c] = preds[c]
    return out

baseline_df = baseline_complete_df(df)
print('baseline missing:', int(baseline_df[option_cols].isna().sum().sum()))

In [ ]:
# ================================================================
# Build pseudo-complete training images from the deterministic baseline.
# This replaces the paper's synthetic Heston-generated complete surfaces
# because this competition gives only dataset.csv.
# ================================================================

complete_images = []
for _, row in baseline_df.iterrows():
    im, _ = row_to_image(row)
    complete_images.append(im)
complete_images = np.stack(complete_images).astype(np.float32)

# normalize using complete training bank
mean_iv = float(complete_images[valid_pixel[None,:,:].astype(bool).repeat(len(complete_images), axis=0)].mean())
std_iv = float(complete_images[valid_pixel[None,:,:].astype(bool).repeat(len(complete_images), axis=0)].std())
std_iv = max(std_iv, 1e-6)
print('mean_iv', mean_iv, 'std_iv', std_iv)

complete_norm = (complete_images - mean_iv) / std_iv
images_norm = (images - mean_iv) / std_iv

plt.figure(figsize=(10,2.5))
plt.imshow(complete_images[0], aspect='auto')
plt.title('Baseline-complete image used as clean training surface')
plt.yticks([0,1], ['CE','PE'])
plt.colorbar();

In [ ]:
# ================================================================
# Train/validation split by timestamp.
# ================================================================

n = len(df)
indices = np.arange(n)
# Keep temporal ordering but random validation subset. For regime check, we also print date split.
rng = np.random.default_rng(SEED)
rng.shuffle(indices)
n_val = max(1, int(VAL_FRAC*n))
val_idx = np.sort(indices[:n_val])
train_idx = np.sort(indices[n_val:])
print('train:', len(train_idx), 'val:', len(val_idx))
print('val date counts:')
print(df.loc[val_idx, 'datetime_parsed'].dt.date.value_counts().sort_index().tail())

In [ ]:
# ================================================================
# NCSN model: small score network with U-Net style skip connections.
# The paper follows U-Net/RefineNet and reduces filters for IV surfaces;
# this implementation uses a tiny shape-safe version for 2 x 14 images.
# ================================================================

class GaussianFourierProjection(nn.Module):
    def __init__(self, embed_dim=64, scale=30.0):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embed_dim//2) * scale, requires_grad=False)
    def forward(self, x):
        x_proj = x[:, None] * self.W[None, :] * 2 * math.pi
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)

class FiLMBlock(nn.Module):
    def __init__(self, channels, emb_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(4, channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(4, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.emb = nn.Linear(emb_dim, channels*2)
    def forward(self, x, emb):
        scale_shift = self.emb(emb).unsqueeze(-1).unsqueeze(-1)
        scale, shift = torch.chunk(scale_shift, 2, dim=1)
        h = self.conv1(F.silu(self.norm1(x)))
        h = h * (1 + scale) + shift
        h = self.conv2(F.silu(self.norm2(h)))
        return x + h

class TinyScoreUNet(nn.Module):
    def __init__(self, in_ch=1, base=64, emb_dim=128):
        super().__init__()
        self.time_embed = nn.Sequential(
            GaussianFourierProjection(64),
            nn.Linear(64, emb_dim), nn.SiLU(),
            nn.Linear(emb_dim, emb_dim), nn.SiLU(),
        )
        self.in_conv = nn.Conv2d(in_ch, base, 3, padding=1)
        self.b1 = FiLMBlock(base, emb_dim)
        self.down_w = nn.Conv2d(base, base, kernel_size=(1,3), stride=(1,2), padding=(0,1))
        self.b2 = FiLMBlock(base, emb_dim)
        self.mid = FiLMBlock(base, emb_dim)
        self.up_w = nn.ConvTranspose2d(base, base, kernel_size=(1,4), stride=(1,2), padding=(0,1))
        self.b3 = FiLMBlock(base, emb_dim)
        self.out_norm = nn.GroupNorm(4, base)
        self.out_conv = nn.Conv2d(base, in_ch, 3, padding=1)
    def forward(self, x, sigma):
        emb = self.time_embed(torch.log(sigma + 1e-12))
        h1 = self.in_conv(x)
        h1 = self.b1(h1, emb)
        h2 = self.down_w(h1)
        h2 = self.b2(h2, emb)
        h = self.mid(h2, emb)
        h = self.up_w(h)
        # Crop/pad to original W.
        if h.shape[-1] > h1.shape[-1]:
            h = h[..., :h1.shape[-1]]
        elif h.shape[-1] < h1.shape[-1]:
            h = F.pad(h, (0, h1.shape[-1]-h.shape[-1]))
        h = h + h1
        h = self.b3(h, emb)
        return self.out_conv(F.silu(self.out_norm(h)))

model = TinyScoreUNet().to(DEVICE)
print(sum(p.numel() for p in model.parameters())/1e6, 'M params')

In [ ]:
# ================================================================
# Noise schedule and dataset for denoising score matching.
# Paper objective: weighted denoising score matching across noise levels.
# Target score for Gaussian perturbation: -(x_tilde - x) / sigma^2.
# ================================================================

train_clean = complete_norm[train_idx]
# empirical sigma max based on pairwise distance scale, clipped for stability
flat = train_clean.reshape(len(train_clean), -1)
if len(flat) > 2 and SIGMA_MAX_MODE == 'data':
    sample = flat[np.random.choice(len(flat), min(256, len(flat)), replace=False)]
    # rough max euclidean / sqrt(D), so per-pixel scale
    dists = []
    for _ in range(512):
        a,b = np.random.choice(len(sample), 2, replace=False)
        dists.append(np.linalg.norm(sample[a]-sample[b]) / math.sqrt(sample.shape[1]))
    sigma_max = float(np.percentile(dists, 95))
    sigma_max = max(sigma_max, 0.5)
else:
    sigma_max = float(SIGMA_MAX_FLOAT)

sigmas = torch.tensor(np.exp(np.linspace(np.log(sigma_max), np.log(SIGMA_MIN), NUM_NOISE_LEVELS)), dtype=torch.float32)
print('sigmas:', sigmas.cpu().numpy())

class ScoreDataset(Dataset):
    def __init__(self, clean_images):
        self.x = torch.tensor(clean_images[:,None,:,:], dtype=torch.float32)
    def __len__(self): return len(self.x)
    def __getitem__(self, idx): return self.x[idx]

loader = DataLoader(ScoreDataset(train_clean), batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def score_matching_loss(model, x, sigmas):
    b = x.shape[0]
    sigma_idx = torch.randint(0, len(sigmas), (b,), device=x.device)
    sigma = sigmas.to(x.device)[sigma_idx].view(b,1,1,1)
    noise = torch.randn_like(x) * sigma
    perturbed = x + noise
    target = -noise / (sigma**2)
    score = model(perturbed, sigma.view(b))
    # Paper weights each noise objective by sigma^2.
    loss = 0.5 * ((score - target)**2 * (sigma**2)).reshape(b, -1).mean(dim=1).mean()
    return loss

In [ ]:
# ================================================================
# Train NCSN
# ================================================================

loss_hist = []
model.train()
for epoch in range(1, EPOCHS+1):
    total = 0.0
    count = 0
    pbar = tqdm(loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)
    for xb in pbar:
        xb = xb.to(DEVICE)
        loss = score_matching_loss(model, xb, sigmas)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += float(loss.item()) * len(xb)
        count += len(xb)
        pbar.set_postfix(loss=float(loss.item()))
    avg = total / max(count,1)
    loss_hist.append(avg)
    if epoch % 10 == 0 or epoch == 1:
        print(f'epoch {epoch}: loss={avg:.6f}')

plt.figure(figsize=(7,3))
plt.plot(loss_hist)
plt.title('Score matching training loss')
plt.xlabel('epoch')
plt.ylabel('loss');

In [ ]:
# ================================================================
# Paper Algorithm 2/3: ALD inpainting.
# For this single-expiry 2x14 image, calendar spread loss is unavailable.
# We use a butterfly/convexity proxy on each row only when USE_NO_ARB_PROXY=True.
# ================================================================

valid_pixel_t = torch.tensor(valid_pixel[None,None,:,:], dtype=torch.float32, device=DEVICE)

def butterfly_proxy_loss_np(img_norm):
    # Convert back to IV scale, use second differences along strikes.
    iv = img_norm * std_iv + mean_iv
    loss = 0.0
    cnt = 0
    for r in range(H):
        y = iv[r, :]
        # Penalize rough negative convexity and excessive roughness gently.
        d2 = y[:-2] - 2*y[1:-1] + y[2:]
        loss += np.mean(np.maximum(-d2, 0.0)**2) + 0.05*np.mean(d2**2)
        cnt += 1
    return float(loss / max(cnt,1))

@torch.no_grad()
def ald_inpaint_one(observed_norm: np.ndarray, known_mask: np.ndarray, restarts=ALD_RESTARTS,
                    use_noarb=USE_NO_ARB_PROXY):
    model.eval()
    known = torch.tensor(observed_norm[None,None,:,:], dtype=torch.float32, device=DEVICE)
    mask = torch.tensor(known_mask[None,None,:,:], dtype=torch.float32, device=DEVICE)
    vpix = valid_pixel_t
    # known values hard-coded only where actual pixels exist and observed
    mask = mask * vpix
    best = None
    best_loss = np.inf
    all_samples = []
    sig = sigmas.to(DEVICE)
    sigma_L = sig[-1]

    for rr in range(restarts):
        x = torch.randn_like(known)
        x = x * (1-mask) + known * mask
        prev_B = 0.0
        prev_C = 0.0
        accepted = 0
        proposed = 0

        for sigma in sig:
            alpha = ALD_EPS * (sigma**2) / (sigma_L**2)
            # Algorithm 2 adds noise to known pixels per noise level.
            known_noisy = known + torch.randn_like(known) * sigma
            for t in range(ALD_STEPS_PER_NOISE):
                z = torch.randn_like(x)
                proposal = x + alpha * model(x, sigma.repeat(1)) + torch.sqrt(2*alpha) * z
                proposal = proposal * (1-mask) + known_noisy * mask
                proposal = proposal * vpix
                proposed += 1

                if use_noarb:
                    B = butterfly_proxy_loss_np(proposal.detach().cpu().numpy()[0,0])
                    C = 0.0
                    beta = max(B/(prev_B + NO_ARB_EPS), C/(prev_C + NO_ARB_EPS))
                    u = random.uniform(0.0, ACCEPT_UPPER)
                    if beta <= u:
                        x = proposal
                        prev_B, prev_C = B, C
                        accepted += 1
                    # else reject: x remains old
                    x = x * (1-mask) + known_noisy * mask
                else:
                    x = proposal
                    accepted += 1

        # Denoising step from the paper.
        x = x + (sigma_L**2) * model(x, sigma_L.repeat(1))
        x = x * (1-mask) + known * mask
        x = x * vpix
        arr = x.detach().cpu().numpy()[0,0]
        arr = np.clip(arr, (1e-6-mean_iv)/std_iv, (1.0-mean_iv)/std_iv)
        loss = butterfly_proxy_loss_np(arr)
        all_samples.append(arr)
        if loss < best_loss:
            best_loss = loss
            best = arr
    # average the samples after rejecting very rough ones; stable for small data.
    samples = np.stack(all_samples)
    return np.median(samples, axis=0).astype(np.float32)

In [ ]:
# ================================================================
# Validation against deterministic baseline.
# We mask known entries from validation complete images, run ALD inpainting,
# and compare only hidden positions. This is the gate that decides final blending.
# ================================================================

def make_validation_masks(true_img, actual_known_mask, repeats=VAL_MASK_REPEATS):
    cases = []
    known_positions = np.argwhere((actual_known_mask > 0) & (valid_pixel > 0))
    if len(known_positions) < 4:
        return cases
    for rep in range(repeats):
        hide = np.zeros_like(actual_known_mask)
        # random holdout from actually observed cells
        k = max(1, int(VAL_HIDE_FRAC * len(known_positions)))
        chosen = known_positions[np.random.choice(len(known_positions), k, replace=False)]
        for i,j in chosen:
            hide[i,j] = 1.0
        cases.append(hide)
        if EDGE_EXTRA_VALIDATION:
            # force edge-like holdout when available
            edge_hide = np.zeros_like(actual_known_mask)
            for r in range(H):
                obs_cols = np.where(actual_known_mask[r] > 0)[0]
                if len(obs_cols) >= 4:
                    edge_hide[r, obs_cols[0]] = 1.0
                    edge_hide[r, obs_cols[-1]] = 1.0
            if edge_hide.sum() > 0:
                cases.append(edge_hide)
    return cases

# baseline prediction under the same artificial masks

def baseline_predict_image_with_mask(row_idx, hidden_mask):
    row = df.loc[row_idx].copy()
    # hide selected originally observed cells
    for c, (i,j) in col_to_pos.items():
        if hidden_mask[i,j] > 0:
            row[c] = np.nan
    preds = baseline_fill_one_row(row)
    img = np.zeros((H,W), dtype=np.float32)
    for c, (i,j) in col_to_pos.items():
        img[i,j] = safe_iv(preds[c])
    return img

val_records = []
max_val_cases = min(len(val_idx), 160)  # keep Kaggle runtime sane
for row_idx in tqdm(val_idx[:max_val_cases], desc='Validating ALD inpainting'):
    true_img = complete_images[row_idx]
    actual_mask = masks[row_idx]
    cases = make_validation_masks(true_img, actual_mask)
    for hidden in cases:
        obs_mask = actual_mask.copy()
        obs_mask[hidden > 0] = 0.0
        observed = true_img.copy()
        observed[obs_mask == 0] = 0.0
        observed_norm = (observed - mean_iv) / std_iv
        observed_norm[obs_mask == 0] = 0.0

        pred_norm = ald_inpaint_one(observed_norm, obs_mask)
        pred_img = pred_norm * std_iv + mean_iv
        base_img = baseline_predict_image_with_mask(row_idx, hidden)
        h = hidden.astype(bool)
        if h.sum() == 0:
            continue
        neural_mse = float(np.mean((pred_img[h] - true_img[h])**2))
        base_mse = float(np.mean((base_img[h] - true_img[h])**2))
        blend_grid = np.linspace(0,1,11)
        best_blend = None
        best_mse = np.inf
        for w in blend_grid:
            bimg = w*pred_img + (1-w)*base_img
            mse = float(np.mean((bimg[h]-true_img[h])**2))
            if mse < best_mse:
                best_mse = mse
                best_blend = float(w)
        date = df.loc[row_idx,'datetime_parsed'].date()
        val_records.append({
            'row_idx': row_idx,
            'date': str(date),
            'n_hidden': int(h.sum()),
            'neural_mse': neural_mse,
            'baseline_mse': base_mse,
            'best_blend_mse': best_mse,
            'best_neural_weight': best_blend,
            'edge_case': bool((hidden[:,0].sum()+hidden[:,-1].sum()) > 0),
        })

val_df = pd.DataFrame(val_records)
val_df.to_csv(f'validation_{OUT_PREFIX}.csv', index=False)
print(val_df.describe(include='all'))
print('\nMean MSE:')
print(val_df[['neural_mse','baseline_mse','best_blend_mse','best_neural_weight']].mean())
print('\nBy edge_case:')
print(val_df.groupby('edge_case')[['neural_mse','baseline_mse','best_blend_mse','best_neural_weight']].mean())

In [ ]:
# ================================================================
# Decide final blend from validation.
# The final system never blindly uses the model: if validation does not beat
# baseline, it falls back to a conservative small blend.
# ================================================================

if len(val_df) == 0:
    print('No validation cases were available. Using conservative neural weight.')
    FINAL_NEURAL_WEIGHT = MAX_NEURAL_BLEND_IF_WORSE
else:
    neural = val_df['neural_mse'].mean()
    base = val_df['baseline_mse'].mean()
    best_blend = val_df['best_neural_weight'].median()
    if neural < base:
        FINAL_NEURAL_WEIGHT = float(np.clip(best_blend, 0.10, 1.00))
        print('Validated neural inpainting is better than baseline.')
    else:
        FINAL_NEURAL_WEIGHT = float(np.clip(best_blend, 0.00, MAX_NEURAL_BLEND_IF_WORSE))
        print('Neural inpainting did NOT beat baseline on mean validation MSE.')
        print('Using conservative blend weight, not full neural output.')

print('FINAL_NEURAL_WEIGHT =', FINAL_NEURAL_WEIGHT)

In [ ]:
# ================================================================
# Final imputation on actual missing cells.
# For each timestamp, run ALD on the actual observed mask, blend with baseline,
# then write only originally missing cells.
# ================================================================

filled = df.copy()
final_rows = []

for row_idx in tqdm(range(len(df)), desc='Final score-based imputation'):
    actual_mask = masks[row_idx].copy()
    if actual_mask.sum() == valid_pixel.sum():
        continue
    observed = images[row_idx].copy()
    observed_norm = (observed - mean_iv) / std_iv
    observed_norm[actual_mask == 0] = 0.0
    pred_norm = ald_inpaint_one(observed_norm, actual_mask)
    pred_img = pred_norm * std_iv + mean_iv

    base_img = np.zeros((H,W), dtype=np.float32)
    base_preds = baseline_fill_one_row(df.loc[row_idx])
    for c, (i,j) in col_to_pos.items():
        base_img[i,j] = safe_iv(base_preds[c])

    final_img = FINAL_NEURAL_WEIGHT * pred_img + (1.0 - FINAL_NEURAL_WEIGHT) * base_img
    final_img = np.maximum(final_img, 1e-6)

    for c, (i,j) in col_to_pos.items():
        if pd.isna(df.at[row_idx,c]):
            filled.at[row_idx,c] = float(final_img[i,j])
            final_rows.append({
                'row_index': row_idx,
                'datetime': df.at[row_idx,'datetime'],
                'contract': c,
                'strike': strike_map[c],
                'option_type': type_map[c],
                'baseline_prediction': float(base_img[i,j]),
                'score_prediction': float(pred_img[i,j]),
                'final_prediction': float(final_img[i,j]),
                'neural_weight': FINAL_NEURAL_WEIGHT,
            })

print('Missing after final:', int(filled[option_cols].isna().sum().sum()))
final_diag = pd.DataFrame(final_rows)
final_diag.head()

In [ ]:
# ================================================================
# Write outputs: filled dataset, submission, diagnostics, model checkpoint.
# ================================================================

filled_out = f'filled_dataset_{OUT_PREFIX}.csv'
submission_out = f'submission_{OUT_PREFIX}.csv'
diag_out = f'diagnostics_{OUT_PREFIX}.csv'
model_out = f'ncsn_model_{OUT_PREFIX}.pt'

filled_save = filled.drop(columns=['datetime_parsed'])
filled_save.to_csv(filled_out, index=False)
final_diag.to_csv(diag_out, index=False)

rows = []
original = df.drop(columns=['datetime_parsed'])
for c in [x for x in original.columns if x != 'datetime']:
    was_missing = original[c].isna()
    for idx in original.index[was_missing]:
        rows.append({'id': f"{original.loc[idx,'datetime']}||{c}", 'value': filled_save.loc[idx,c]})
submission = pd.DataFrame(rows).sort_values('id').reset_index(drop=True)
submission.to_csv(submission_out, index=False)

torch.save({
    'model_state_dict': model.state_dict(),
    'mean_iv': mean_iv,
    'std_iv': std_iv,
    'sigmas': sigmas.cpu().numpy(),
    'final_neural_weight': FINAL_NEURAL_WEIGHT,
    'H': H,
    'W': W,
    'col_to_pos': col_to_pos,
}, model_out)

print('Saved:')
print(' ', filled_out)
print(' ', submission_out, len(submission), 'rows')
print(' ', diag_out)
print(' ', model_out)
submission.head()

## What to check after running

1. Open `validation_score_ncsn_iv.csv`.
2. Compare `neural_mse`, `baseline_mse`, and `best_blend_mse`.
3. If the notebook prints **“Validated neural inpainting is better than baseline”**, then the paper-style score model was useful on your synthetic holdout masks.
4. If it does not beat baseline, do not trust the neural model blindly. The notebook will automatically use a conservative blend.

For serious Kaggle runs, increase `EPOCHS`, `ALD_RESTARTS`, and `ALD_STEPS_PER_NOISE`, then re-check validation. The method is stochastic, so keep the validation gate.